# Find the baseline epoch count

This notebook trains `TinyCNN` for up to 30 epochs and uses validation loss to choose the best epoch. It uses training and validation data only. The test split stays untouched.

## Colab setup

1. Select **Runtime > Change runtime type > GPU**.
2. Make this project available in Colab.
3. Copy or mount the ignored `frames/` directory inside the project directory.
4. Set `PROJECT_DIR` below, then run the notebook from top to bottom.

The repository dependencies must already be available in the Colab environment.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/movie_classifier")
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

assert Path("train_baseline.py").exists(), "Set PROJECT_DIR to the repository directory."
assert next(Path("frames").glob("*/*.jpg"), None), "Put the movie frames in PROJECT_DIR/frames."
print(f"project_dir={PROJECT_DIR}")

In [ ]:
import copy

import matplotlib.pyplot as plt
import torch
from torch import nn

from dataset import CLASSES
from train_baseline import TinyCNN, build_dataloaders, evaluate, train_epoch

MAX_EPOCHS = 30
PATIENCE = 5
MIN_DELTA = 0.001
LEARNING_RATE = 1e-3
SEARCH_CHECKPOINT = Path("baseline_epoch_search_best.pt")

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch={torch.__version__} device={device}")

## Build the split once

This can take a couple of minutes because `build_splits()` checks perceptual similarity across all frames. No test loader is created.

In [ ]:
train_loader, val_loader = build_dataloaders()
print(
    f"train_samples={len(train_loader.dataset)} "
    f"val_samples={len(val_loader.dataset)} classes={len(CLASSES)}"
)

## Train with early stopping

The best epoch is the one with the lowest validation loss. Training stops after five epochs without a meaningful improvement. The best weights are kept separately from `baseline_model.pt`.

In [ ]:
model = TinyCNN(len(CLASSES)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

history = []
best_epoch = 0
best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_accuracy = train_epoch(
        model, train_loader, optimizer, loss_fn, device
    )
    val_loss, val_accuracy = evaluate(model, val_loader, loss_fn, device)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        }
    )
    print(
        f"epoch {epoch}: train_loss={train_loss:.3f} "
        f"train_acc={train_accuracy:.3f} val_loss={val_loss:.3f} "
        f"val_acc={val_accuracy:.3f}",
        flush=True,
    )

    if val_loss < best_val_loss - MIN_DELTA:
        best_epoch = epoch
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"early stopping after epoch {epoch}")
        break

torch.save(best_state, SEARCH_CHECKPOINT)
print(f"suggested_epochs={best_epoch}")
print(f"best_val_loss={best_val_loss:.3f}")
print(f"saved={SEARCH_CHECKPOINT}")

## Inspect the learning curves

A growing gap where training keeps improving while validation loss rises is evidence of overfitting. The dotted line marks the suggested epoch count.

In [ ]:
epochs = [row["epoch"] for row in history]
train_losses = [row["train_loss"] for row in history]
val_losses = [row["val_loss"] for row in history]
train_accuracies = [row["train_accuracy"] for row in history]
val_accuracies = [row["val_accuracy"] for row in history]

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, train_losses, marker="o", label="train")
axes[0].plot(epochs, val_losses, marker="o", label="validation")
axes[0].axvline(best_epoch, color="black", linestyle=":", label="best epoch")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[0].legend()

axes[1].plot(epochs, train_accuracies, marker="o", label="train")
axes[1].plot(epochs, val_accuracies, marker="o", label="validation")
axes[1].axvline(best_epoch, color="black", linestyle=":", label="best epoch")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
axes[1].legend()

figure.tight_layout()
figure.savefig("baseline_epoch_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved=baseline_epoch_curves.png")

## Interpret the result

- Use `suggested_epochs` as the next baseline epoch count.
- If validation loss is still falling at epoch 30, increase `MAX_EPOCHS` and rerun.
- If validation loss rises while training loss falls, later epochs are overfitting.
- Do not use test accuracy to adjust the epoch count. Test data should be evaluated only after model choices are finished.